# XOR: Validação da MLP

Antes de aplicar a rede neural ao conjunto MNIST, a implementação será validada utilizando o problema XOR.

O XOR é um exemplo clássico utilizado para testar redes neurais porque seus dados não são linearmente separáveis. Isso significa que um perceptron simples não consegue resolver o problema corretamente, sendo necessária pelo menos uma camada oculta.

Como o conjunto possui apenas quatro exemplos, ele permite verificar com facilidade se as etapas de propagação direta (forward propagation), cálculo dos erros e retropropagação dos gradientes (backpropagation) foram implementadas corretamente.

Se a rede conseguir aprender o XOR e atingir uma boa taxa de acerto, haverá maior confiança de que a implementação está funcionando corretamente antes de avançar para um problema mais complexo como o MNIST.

## Etapa 1: Carregar XOR

In [1]:
import numpy as np

# 4 exemplos do XOR
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
])

y = np.array([[0], [1], [1], [0]])

print("X shape:", X.shape) 
print("y shape:", y.shape) 
print("\nDados:")
for i in range(len(X)):
    print(f"  {X[i]} → {y[i][0]}")

X shape: (4, 2)
y shape: (4, 1)

Dados:
  [0 0] → 0
  [0 1] → 1
  [1 0] → 1
  [1 1] → 0


## Etapa 2: Função de ativação (Sigmoid)

Para este experimento foi utilizada a função de ativação Sigmoid, que transforma qualquer valor real em um número entre 0 e 1. Essa característica é útil em problemas de classificação binária, como o XOR.

Fórmula: $σ(x) = \frac{1}{1 + e^{-x}}$


Uma vantagem da Sigmoid é que sua derivada pode ser calculada a partir do próprio valor da função, sem necessidade de operações adicionais complexas. Isso simplifica a implementação do algoritmo de backpropagation, utilizado para ajustar os pesos da rede durante o treinamento.

In [2]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    s = sigmoid(x)
    return s * (1 - s)

# Teste rápido
test_values = np.array([-2, -1, 0, 1, 2])
print("x:          ", test_values)
print("sigmoid(x): ", sigmoid(test_values).round(4))
print("derivada:   ", sigmoid_derivative(test_values).round(4))

x:           [-2 -1  0  1  2]
sigmoid(x):  [0.1192 0.2689 0.5    0.7311 0.8808]
derivada:    [0.105  0.1966 0.25   0.1966 0.105 ]


## Etapa 3: Inicialização dos pesos

Cada camada tem dois parâmetros: W (pesos) e b (bias).

- W conecta os neurônios entre camadas: `shape (neurônios_entrada, neurônios_saída)`
- b é somado depois da multiplicação: `shape (1, neurônios_saída)`

Inicializamos W com valores aleatórios pequenos diferentes de zero.
Se todos os pesos fossem iguais, todos os neurônios aprenderiam a mesma coisa ("problema de simetria"). Valores aleatórios quebram essa simetria.

Os bias foram inicializados com zero, pois o problema de simetria ocorre nos pesos das conexões entre neurônios. Como cada neurônio possui pesos diferentes, os bias podem começar em zero sem prejudicar o aprendizado.

In [3]:
np.random.seed(42)  # reprodutibilidade

# Camada 1: 2 entradas → 2 neurônios ocultos
W1 = np.random.randn(2, 2) * 0.1
b1 = np.zeros((1, 2))

# Camada 2: 2 neurônios ocultos → 1 saída
W2 = np.random.randn(2, 1) * 0.1
b2 = np.zeros((1, 1))

print("W1 shape:", W1.shape, "\n", W1)
print("\nb1 shape:", b1.shape, "\n", b1)
print("\nW2 shape:", W2.shape, "\n", W2)
print("\nb2 shape:", b2.shape, "\n", b2)

W1 shape: (2, 2) 
 [[ 0.04967142 -0.01382643]
 [ 0.06476885  0.15230299]]

b1 shape: (1, 2) 
 [[0. 0.]]

W2 shape: (2, 1) 
 [[-0.02341534]
 [-0.0234137 ]]

b2 shape: (1, 1) 
 [[0.]]


## Etapa 4: Forward Pass

O forward pass representa o fluxo de informações da entrada até a saída da rede neural.

Em cada camada são realizadas duas etapas:

1. Aplicação da transformação linear: z = XW + b
2. Aplicação da função de ativação: a = sigmoid(z)

Os valores z são chamados de pré-ativações, enquanto os valores a representam as ativações dos neurônios e são utilizados como entrada para a próxima camada.

Ao final do processo, a rede produz uma predição para cada exemplo do conjunto XOR.

In [5]:
def forward(X, W1, b1, W2, b2):
    # Camada 1
    z1 = X @ W1 + b1      # (4,2) @ (2,2) = (4,2)
    a1 = sigmoid(z1)       # (4,2)
    
    # Camada 2 (saída)
    z2 = a1 @ W2 + b2     # (4,2) @ (2,1) = (4,1)
    a2 = sigmoid(z2)       # (4,1)
    
    return z1, a1, z2, a2

z1, a1, z2, a2 = forward(X, W1, b1, W2, b2)

print("z1 shape:", z1.shape)
print("a1 shape:", a1.shape)
print("z2 shape:", z2.shape)
print("a2 shape:", a2.shape)
print("\nPredições iniciais:")
print(a2.round(4))
print("\nEsperado:")
print(y)

z1 shape: (4, 2)
a1 shape: (4, 2)
z2 shape: (4, 1)
a2 shape: (4, 1)

Predições iniciais:
[[0.4941]
 [0.4938]
 [0.4941]
 [0.4938]]

Esperado:
[[0]
 [1]
 [1]
 [0]]
